# Big Iron

See <a href="https://www.kaggle.com/competitions/bluebook-for-bulldozers/leaderboard">Blue Book for Bulldozers</a> leaderboard page.

BTW this notebook borrows heavily from <a href="https://github.com/fastai/fastbook/blob/master/09_tabular.ipynb">chapter 9</a> of
<a href="https://github.com/fastai/fastbook">Fastai's FastBook.
    
<mark>Fastai also has a set of <a href="https://www.fast.ai/posts/2022-07-21-dl-coders-22.html">online courses</a> and librarys (see below) designed to show regular people how to apply advanced machine and deep learning algorithms to real world problems.  Its extremely applied and fortunately, much of the complexity is handled by the library itself.  
    
<b>It is the best course I know of for teaching AI, Machine Learning and Data Science.

In [23]:
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import numpy as np
# Set max rows and columns displayed in jupyter
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

#the following gives access to utils folder
#where utils package stores shared code
import os
import sys
PROJECT_ROOT = os.path.abspath(os.path.join(
                  os.getcwd(),
                  os.pardir)
)

#only add it once
if (PROJECT_ROOT not in sys.path):
    sys.path.append(PROJECT_ROOT)
    
import utils as ut

#want to use a subset of the data?
use_latest=False
latest_size=200000

<mark>install fastai

In [24]:
#install fastai library
# !pip install fastai
#or see https://pypi.org/project/fastai/

# Get the data

In [25]:
#install kaggle api
# !pip install kaggle

In [29]:
!cd ./data;ls -la

total 174748
drwxrwxr-x 2 keith keith      4096 Mar 16  2023  .
drwxrwxr-x 6 keith keith      4096 Mar 12 12:13  ..
-rw-rw-r-- 1 keith keith     11063 Dec 11  2019 'Data Dictionary.xlsx'
-rw-rw-r-- 1 keith keith  51498702 Dec 11  2019  Machine_Appendix.csv
-rw-rw-r-- 1 keith keith    196760 Dec 11  2019  median_benchmark.csv
-rw-rw-r-- 1 keith keith    211941 Dec 11  2019  random_forest_benchmark_test.csv
-rw-rw-r-- 1 keith keith   3560907 Dec 11  2019  Test.csv
-rw-rw-r-- 1 keith keith 119791159 Dec 11  2019  TrainAndValid.csv
-rw-rw-r-- 1 keith keith   3318969 Dec 11  2019  Valid.csv
-rw-rw-r-- 1 keith keith    323524 Dec 11  2019  ValidSolution.csv


In [26]:
#if you have the Kaggle api installed then use it
!kaggle competitions download -c bluebook-for-bulldozers


bluebook-for-bulldozers.zip: Skipping, found more recently modified local copy (use --force to force download)


In [4]:
#uhoh...my key is exposed, get your key from kaggle place it in the noted folder
!chmod 600 /home/keith/.kaggle/kaggle.json

In [3]:
!ls

archive			     data
bigiron1.ipynb		     df.csv
bigiron-pipeline.ipynb	     fed-funds-rate-historical-chart.csv
bluebook-for-bulldozers.zip  vix-volatility-index-historical-chart.csv
catboost_info		     wheat-prices-historical-chart-data.csv


# EDA  
git restore --staged homework/data/TrainAndValid.csv

#start here 3/19/26

In [30]:
df=pd.read_csv('./data/TrainAndValid.csv')
print(df.shape)

(412698, 53)


/tmp/ipykernel_14225/1701633161.py:1: DtypeWarning: Columns (13,39,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('./data/TrainAndValid.csv')


In [31]:
# df.dtypes
df.shape

(412698, 53)

In [12]:
df.Blade_Extension.unique()

array([nan, 'Yes', 'None or Unspecified'], dtype=object)

In [13]:
#Is a column mostly full of NaN's?
for c in df.columns:
    if df[c].isnull().sum()/len(df) > 0.9:
        print(f"Column {c} has more than 90% missing values.")
        # df.drop(c, axis=1, inplace=True)

Column Blade_Extension has more than 90% missing values.
Column Blade_Width has more than 90% missing values.
Column Enclosure_Type has more than 90% missing values.
Column Engine_Horsepower has more than 90% missing values.
Column Pushblock has more than 90% missing values.
Column Scarifier has more than 90% missing values.
Column Tip_Control has more than 90% missing values.


In [14]:
df.describe()

,SalesID,SalePrice,MachineID,ModelID,datasource,auctioneerID,YearMade,MachineHoursCurrentMeter
count,4.126980e+05,412698.000000,4.126980e+05,412698.000000,412698.000000,392562.000000,412698.000000,1.475040e+05
mean,2.011161e+06,31215.181414,1.230061e+06,6947.201828,135.169361,6.585268,1899.049637,3.522988e+03
std,1.080068e+06,23141.743695,4.539533e+05,6280.824982,9.646749,17.158409,292.190243,2.716993e+04
min,1.139246e+06,4750.000000,0.000000e+00,28.000000,121.000000,0.000000,1000.000000,0.000000e+00
25%,1.421898e+06,14500.000000,1.088593e+06,3261.000000,132.000000,1.000000,1985.000000,0.000000e+00
50%,1.645852e+06,24000.000000,1.284397e+06,4605.000000,132.000000,2.000000,1995.000000,0.000000e+00
75%,2.261012e+06,40000.000000,1.478079e+06,8899.000000,136.000000,4.000000,2001.000000,3.209000e+03
max,6.333349e+06,142000.000000,2.486330e+06,37198.000000,173.000000,99.000000,2014.000000,2.483300e+06


In [19]:
len(df[df.YearMade ==1000])

39391

### Note the sale date, this is a time series problem, that changes how we split the data into train and validation set (cannot use random splitter)

To split: Older data is training data, Newer data is validation data.  This keeps information from leaking from the validation set back to the training set.

### Lets see what the date range is

In [20]:
df['saledate']=pd.to_datetime(df['saledate'])

In [21]:
print(f'earliest date= {df.saledate.min()}, latest date={df.saledate.max()}')

earliest date= 1989-01-17 00:00:00, latest date=2012-04-28 00:00:00


### Sort the dataframe by saledate in preparation for splitting into train and validation sets
<mark> train on oldest, test on newest

In [22]:
#how muchmemory am I using?
import psutil

process = psutil.Process()
memory_usage = process.memory_info().rss / (1024 ** 2)  # Convert bytes to MB
print(f"Memory usage: {memory_usage:.2f} MB")

Memory usage: 518.54 MB


In [ ]:
df.sort_values(by=['saledate'], inplace=True)

#want to use a subset?
if(use_latest):
    df=df[-latest_size:]
len(df)

### Add a Time field for the Linear regressor that we will use to subtract the trend

In [ ]:
dates=pd.DatetimeIndex(df.saledate)
df['Time']=[(dates[i]-dates[0]).days for i in range(len(df))]
df['Time']
# df[['saledate','Time']].head(20)

## <mark>Hmmm.. this dataset has sales around 2007, when the almost great depression hit

I wonder if a vix entry would help?  Goto <a href="https://www.macrotrends.net/2603/vix-volatility-index-historical-chart">VIX Volatility Index - Historical Chart</a> to grab this data, get all years and download the csv.  See if it helps, if so keep it, if not dont.
    
While we are at it. Lets get a few more: 
    
1.<a href="https://www.macrotrends.net/2534/wheat-prices-historical-chart-data">Wheat Prices - 40 Year Historical Chart</a><br>
2.<a href="https://www.macrotrends.net/2015/fed-funds-rate-historical-chart">Federal Funds Rate - 62 Year Historical Chart</a>  

In [ ]:
fed=pd.read_csv('fed-funds-rate-historical-chart.csv',skiprows=15)
# print(f" length={len(fed)}, NaNs in value={fed.loc[:,' value'].isnull().sum()}")
print(f" length={len(fed)}, NaNs in value={fed.loc[:-21,' value'].isnull().sum()}")

In [ ]:
#create a little function to preprocess, scale and add to df
from sklearn.preprocessing import StandardScaler
def get_df(df,filename,new_col_name):
    dfn=pd.read_csv(filename,skiprows=15)
    dfn['date'] =  pd.to_datetime(dfn['date'])
    dfn.rename(columns={' value':new_col_name},inplace=True)
    
    #scale it
    dfn[new_col_name]=StandardScaler().fit_transform(dfn[[new_col_name]])
    
    #now merge
    df = pd.merge(df,dfn, left_on='saledate', right_on='date', how='left')
    df.drop(columns=['date'],inplace=True)
    df[new_col_name].fillna(method='backfill',inplace=True)
    df[new_col_name].fillna(method='ffill',inplace=True)
    return df

#the following 2 did not help, the last one did though
# df=get_df(df,'./vix-volatility-index-historical-chart.csv','vix')
# df=get_df(df,'./wheat-prices-historical-chart-data.csv','wheat')
# get_df(df,'./fed-funds-rate-historical-chart.csv','fedrate')
df=get_df(df,'./fed-funds-rate-historical-chart.csv','fedrate')
# df

## <mark>Split Dates into more useful features

Its hard for a random forest to use a datetime object since it has a lot of encoded information (the year,the month, the day of the week, weekday, weekend, holiday, end of quarter etc.).  We can slog through and manually create these features, or use fastai, a library that already does this.

In [ ]:
orig=list(df.columns)

In [ ]:
#using a fastai helper function to get ALL the date info
from fastai.tabular import core
df = core.add_datepart(df, 'saledate')

#look at all those additional sale columns
[col for col in df.columns if 'sale' in col]

In [ ]:
new=list(df.columns)
[col for col in new if col not in orig]

Any obvious outliers?

In [ ]:
df.describe()

In [ ]:
df.saleYear.max()
df[df.YearMade<1900]

### <mark>YearMade has some tractors made in the year 1000, how to fix this?

In [ ]:
sns.histplot(df,x='YearMade')

## Lets try to infer bogus YearMades (==1000)

In [ ]:
#how many made before 1300?
df.loc[df.YearMade==1000].shape[0]

In [ ]:
for c in df.columns:
    if df[c].isnull().sum()==0:
        print(f'{c}: {df[c].nunique()}')

In [ ]:
#look for good columns to groupby, dont use ones with a lot of NaNs or too many/few values
# df.nunique().sort_values()
# df.select_dtypes(include=['object'])
# df.select_dtypes(include=['object']).dtypes

In [ ]:
#find good columns to groupby
tmp=df.loc[:,['fiModelDesc','ModelID','fiBaseModel','fiModelSeries','fiModelDescriptor','fiProductClassDesc']]

# #want few nulls
tmp.isnull().sum()

In [ ]:
df.ModelID.nunique()

In [ ]:
#ModelID has 5281 unique values and no NaNs, we have a winner
#IMPORTANT eliminate the bogus values first (everything < 1800) or we will skew the groupby
gb=df[df.YearMade>1800].groupby(["ModelID"]).YearMade.mean()
# gb.value_counts()

gb

In [ ]:
#how to select from the groupby
gb.iloc[0]

Lets create a function to use the groupby above to infer the missing YearMades, just assumme average

In [ ]:
# %%time
# #naive attempt
# #OK lets create a function to try to replace yearmades=1000 using above group by
# def fun(row):
#     if(row.YearMade==1000):
#         # print(row.ModelID)
#         try:
#             #see if we have an entry for this model
#             row.YearMade=gb[row.ModelID]
#         except KeyError as e:
#             pass
#     return row
# df.apply(fun,axis=1)

## I stopped the above cell after 10 minutes, too slow on a half million rows, try vectorization

In [ ]:
#lects vectorize
#if we use vectorize, we need to map the column name to the number
dct={c:i for i,c in enumerate(df.columns)}
dct

In [ ]:
%%time
def fun_vectorized(row):
    if(row[dct['YearMade']]==1000):
        try:
            row[dct['YearMade']]=gb[row[dct['ModelID']]]
        except KeyError as e:
            pass
    return row
df=df.apply(fun_vectorized,axis=1,raw=True)  #the raw does it

In [ ]:
#went from 39000 to 764 or so, not bad on a 1/2 million row dataset
(df.YearMade==1000).sum()

### Maybe we can use groupby to infer the incorrect YearMades?

In [ ]:
def impute_YearMade(df,year=1950):
    '''
    replace any YearMade<year with the max for the below groupby or year

    df: dataframe to impute YearMade on
    year: anything below this year is imputed, if cannot be imputed set to year
    '''
    #lets try fiBaseModel and fiProductClassDesc for mean YearBuilt
    estimates=df.groupby(['fiModelDesc','fiProductClassDesc']).YearMade.mean()
    
    #for a given row, look up and return the mean YearMade
    def impute_yb(x):
        est=estimates[(x['fiModelDesc'],  x['fiProductClassDesc'])]
        if est is np.nan or est < year:
            est=year
        return est
  
    df['YearMade']=df.apply(lambda x:impute_yb(x) if x.YearMade<year else x.YearMade,axis=1)   
    return df

In [ ]:
%%time
df=impute_YearMade(df)

### See what the most sales dates are

In [ ]:
df.saleYear.hist()

### Maybe the older data is not as relevant?  Drop everything <2004

Also not as much of it

In [ ]:
df

In [ ]:
filt=df['saleYear']>2004
# filt=df['Time']>7300  #just the rows where fed fund rate is lowest, killed performence RMSLE went to .24
df=df[filt]
df.reset_index(inplace=True)

In [ ]:
# df.to_csv('df.csv',index=False)
# df=pd.read_csv('df.csv')

In [ ]:
df.shape

## Prepare the rest of the data (the super easy way)

Use FastAI's TabularPandas class to prepare data

### Dependant var

We are trying to predict 'SalePrice'.  Specifically the root mean squared log error (RMSLE) between the actual and predicted values.  Start by taking the log of the dependant variable (the L of RMSLE) 

start here 4/23/25

In [ ]:
dep_var = 'SalePrice'
#take the log of dependant var since that is what the contest wants
df[dep_var] = np.log(df[dep_var].astype(float))

In [ ]:
df['SalePrice']=df['SalePrice'].astype(float)

## Get the indexes of train and test

Its a time series, the latest entries are the test set, the earlier ones, the train set.  Everything later than Oct 2010 is test

In [ ]:
len(df)

In [ ]:
(df['Time']>7300).sum()

In [ ]:
# df=df[df.Time>7300]

In [ ]:
#what if we just use 2012 as the test?
df.saleYear.value_counts()[2012]/len(df)*100

In [ ]:
cond = (df.saleYear<2012 ) 
train_idx = np.where( cond)[0]
valid_idx = np.where(~cond)[0]

splits = (list(train_idx),list(valid_idx))

### What do they look like BTW

Lets also see what the vix and some commodity prices are

In [ ]:
# trn_subset=train_idx[-500:]
# valid_subset=valid_idx[:500]
trn_subset=train_idx
valid_subset=valid_idx
trn_and_valid_subset=np.append(trn_subset,valid_subset)

fig, (ax1,ax4) = plt.subplots(nrows=2,figsize=(14,6))

# the following 2 take a long time to plot we only want to see where they begin and end
trn_subset = np.random.choice(trn_subset, 1000)
valid_subset=np.random.choice(valid_subset, 1000)
_=sns.scatterplot(x=df.iloc[trn_subset].Time, y=df.iloc[trn_subset].SalePrice,ax=ax1,  marker="o")
_=sns.scatterplot(x=df.iloc[valid_subset].Time, y=df.iloc[valid_subset].SalePrice,ax=ax1,  marker="+")

# _=sns.scatterplot(x=df.iloc[trn_and_valid_subset].Time, y=df.iloc[trn_and_valid_subset].vix,ax=ax2,  marker="+")
# _=sns.scatterplot(x=df.iloc[trn_and_valid_subset].Time, y=df.iloc[trn_and_valid_subset].wheat,ax=ax3,  marker="o")
_=sns.scatterplot(x=df.iloc[trn_and_valid_subset].Time, y=df.iloc[trn_and_valid_subset].fedrate,ax=ax4,  marker="o")